# Generative Models
## Learning and Sampling from Probability Distributions

**MAT 4953 / MAT 6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/generative_models.ipynb)

---

Every generative model answers the same fundamental question: **how do we learn a probability distribution $p_\theta(\mathbf{x})$ from data, and then sample new data from it?**

This notebook surveys the mathematical ideas behind three major families of generative models:

1. **Linear factor models** (probabilistic PCA) — the simplest latent-variable model: a linear map from a low-dimensional Gaussian to data space.
2. **Variational Autoencoders (VAEs)** — nonlinear latent-variable models trained via the evidence lower bound (ELBO).
3. **Autoregressive models & LLMs** — factoring $p(\mathbf{x})$ via the chain rule into a product of conditional distributions.

The Transformer architecture (introduced in the [Architectures Overview](dnn_architectures_overview.ipynb)) serves as the backbone of modern LLMs and bridges the transition from latent-variable to autoregressive generation.

**Prerequisites:** Familiarity with feedforward networks, autoencoders (Sections 2 and 6 of the [Architectures Overview](dnn_architectures_overview.ipynb)), and basic probability (Bayes' rule, Gaussian distributions, KL divergence).

---
# 0. Setup

This notebook runs on **Google Colab** (GPU/TPU/CPU) and in a **local Jupyter environment** (CPU).

- **Google Colab**: click *Open in Colab* above. Go to *Runtime → Change runtime type* and select
  **GPU** or **TPU v2** for faster training.
- **Local**: install dependencies with `pip install -r requirements.txt` from the repository root,
  then open this file with `jupyter notebook` or `jupyter lab`.

The setup cell below automatically detects the available hardware and configures JAX accordingly.

In [ ]:
# @title Runtime & backend setup (run first)
import os, subprocess, platform

def _detect_platform():
    try:
        import requests
        resp = requests.get(
            'http://metadata.google.internal/computeMetadata/v1/instance/attributes/tpu-name',
            headers={'Metadata-Flavor': 'Google'}, timeout=2)
        if resp.ok:
            return 'tpu'
    except Exception:
        pass
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5)
        if result.returncode == 0 and result.stdout.strip():
            return 'gpu'
    except Exception:
        pass
    if platform.system() == 'Darwin':
        return 'cpu_macos'
    return 'cpu'

_platform = _detect_platform()
print(f'Platform: {_platform.upper()}')

os.environ['KERAS_BACKEND'] = 'jax'
if _platform == 'tpu':
    pass
elif _platform == 'gpu':
    os.environ['JAX_PLATFORMS'] = 'cuda'
else:
    os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from IPython.display import display, HTML

import jax
import jax.numpy as jnp
import keras
from keras import layers, ops

%matplotlib inline
np.random.seed(42)
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})

print(f'JAX devices : {jax.devices()}')
print(f'Keras version: {keras.__version__}')
print(f'Keras backend: {keras.backend.backend()}')

---
# 1. The Generative Modeling Problem

## 1.1 Discriminative vs. generative models

Most models we have studied so far are **discriminative**: given an input $\mathbf{x}$, they predict a label $y$ by learning $p(y \mid \mathbf{x})$. A **generative model** instead learns the distribution of the data itself:

$$\boxed{\text{Goal: learn } p_\theta(\mathbf{x}) \approx p_{\text{data}}(\mathbf{x}).}$$

Once we have $p_\theta$, we can:
- **Sample** new data points $\mathbf{x}_{\text{new}} \sim p_\theta(\mathbf{x})$.
- **Evaluate likelihood**: score how probable any given datum is.
- **Interpolate** in a learned latent space between data points.

## 1.2 Three strategies for learning distributions

| Strategy | Key idea | Examples |
|---|---|---|
| **Latent variable** | Introduce hidden variables $\mathbf{z}$; define $p_\theta(\mathbf{x}) = \int p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})\, d\mathbf{z}$ | PCA, VAE, diffusion models |
| **Autoregressive** | Factor via the chain rule: $p(\mathbf{x}) = \prod_t p(x_t \mid x_{<t})$ | RNN language models, GPT, LLMs |
| **Implicit** | Learn a map $G: \mathbf{z} \mapsto \mathbf{x}$ without explicit density | GANs |

This lecture focuses on the first two strategies, which admit clean mathematical formulations.

---
# 2. Linear Factor Models: From PCA to Probabilistic PCA

## 2.1 PCA as a generative model

Recall from the [NumPy & SVD tutorial](numpy_tutorial_svd.ipynb) that PCA finds a $k$-dimensional subspace that captures the most variance. We can reinterpret PCA probabilistically:

$$\mathbf{z} \sim \mathcal{N}(\mathbf{0}, I_k), \qquad \mathbf{x} = W\mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 I_d),$$

where $W \in \mathbb{R}^{d \times k}$ is the *factor loading matrix* and $\boldsymbol{\mu}$ is the data mean.

This is **probabilistic PCA** (Tipping & Bishop, 1999). The marginal distribution of $\mathbf{x}$ is:

$$p(\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu},\; WW^\top + \sigma^2 I_d).$$

The key limitation is that the map from $\mathbf{z}$ to $\mathbf{x}$ is **linear**: the model can only capture correlations that live on a flat subspace. To model curved manifolds, we need a nonlinear decoder — which leads us to VAEs.

In [ ]:
# Generate data on a noisy 2-D circle (a curved 1-D manifold)
n_pts = 500
theta = np.random.uniform(0, 2 * np.pi, n_pts)
X_circle = np.stack([np.cos(theta), np.sin(theta)], axis=1).astype(np.float32)
X_circle += np.random.randn(n_pts, 2).astype(np.float32) * 0.08

# Probabilistic PCA: fit via SVD
mu = X_circle.mean(axis=0)
X_centered = X_circle - mu
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
W_pca = Vt[:1].T * S[0] / np.sqrt(n_pts)  # k=1 component

# Sample from probabilistic PCA
z_samples = np.random.randn(300, 1).astype(np.float32)
X_pca_gen = z_samples @ W_pca.T + mu

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(X_circle[:, 0], X_circle[:, 1], c=theta, cmap='hsv',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[0].set_title('Real Data (circle)', fontweight='bold')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_pca_gen[:, 0], X_pca_gen[:, 1], c='steelblue',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[1].set_title('Probabilistic PCA Samples (k=1)', fontweight='bold')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(axes[0].get_xlim()); axes[1].set_ylim(axes[0].get_ylim())

# Show the linear subspace
t_line = np.linspace(-3, 3, 100)
line_pts = np.outer(t_line, W_pca.flatten()) + mu
axes[2].scatter(X_circle[:, 0], X_circle[:, 1], c=theta, cmap='hsv',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.4)
axes[2].plot(line_pts[:, 0], line_pts[:, 1], 'r-', linewidth=2.5,
             label='PCA subspace')
axes[2].set_title('PCA Captures a Line, Not a Circle', fontweight='bold')
axes[2].set_aspect('equal'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Probabilistic PCA generates points along a LINE — it cannot capture curvature.')

The figure above makes the limitation clear: probabilistic PCA generates data along a **straight line** through the data cloud. To capture the circular manifold, we need a **nonlinear** decoder.

---
# 3. Variational Autoencoders (VAEs)

## 3.1 From autoencoders to generative models

Recall from the [Architectures Overview](dnn_architectures_overview.ipynb) that an autoencoder learns an encoder $E$ and decoder $D$ to minimize reconstruction error. But a plain autoencoder does **not** define a probability distribution — we cannot sample from it meaningfully because the latent space may have "holes" (regions that never arise from encoding real data).

A **Variational Autoencoder** (Kingma & Welling, 2014) solves this by:
1. Making the encoder **stochastic**: instead of mapping $\mathbf{x} \mapsto \mathbf{z}$, it maps $\mathbf{x} \mapsto (\boldsymbol{\mu}, \boldsymbol{\sigma}^2)$, parameterizing a Gaussian $q_\phi(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$.
2. Adding a **KL-divergence regularizer** that pushes $q_\phi(\mathbf{z} \mid \mathbf{x})$ toward the prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, I)$.

## 3.2 The Evidence Lower Bound (ELBO)

We want to maximize $\log p_\theta(\mathbf{x})$ but this involves an intractable integral over $\mathbf{z}$. Instead, we maximize a **lower bound**:

$$\boxed{\log p_\theta(\mathbf{x}) \;\geq\; \underbrace{\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}\!\bigl[\log p_\theta(\mathbf{x} | \mathbf{z})\bigr]}_{\text{reconstruction}} \;-\; \underbrace{D_{\text{KL}}\!\bigl(q_\phi(\mathbf{z}|\mathbf{x}) \,\|\, p(\mathbf{z})\bigr)}_{\text{regularization}} \;=\; \text{ELBO}(\theta, \phi; \mathbf{x}).}$$

**Why this works:**
- The **reconstruction term** encourages the decoder to produce data that looks like the input (same as an autoencoder).
- The **KL term** ensures the latent codes stay close to the standard Gaussian prior, preventing holes and enabling smooth interpolation.

For Gaussian $q_\phi(\mathbf{z} | \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ and prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, I)$, the KL divergence has a closed-form expression:

$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^{k} \bigl(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\bigr).$$

## 3.3 The reparameterization trick

To backpropagate through the stochastic sampling $\mathbf{z} \sim \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$, we rewrite it as a **deterministic** function of a noise variable:

$$\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, I), \qquad \mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}.$$

Now the gradient $\nabla_\phi \mathbf{z}$ is well-defined (it flows through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$), and we can train the encoder and decoder jointly with standard backpropagation.

In [ ]:
# Diagram: VAE architecture with the reparameterization trick
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(-0.5, 6.5); ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Variational Autoencoder — Data Flow', fontsize=14, fontweight='bold')

# Boxes
boxes = [
    (0, 1, '$\\mathbf{x}$', '#f9e79f'),
    (1.5, 1, 'Encoder\n$q_\\phi(\\mathbf{z}|\\mathbf{x})$', '#fadbd8'),
    (3, 1.7, '$\\boldsymbol{\\mu}$', '#d5f5e3'),
    (3, 0.3, '$\\log\\boldsymbol{\\sigma}^2$', '#d5f5e3'),
    (4.2, 1, '$\\mathbf{z}$', '#aed6f1'),
    (5.5, 1, 'Decoder\n$p_\\theta(\\mathbf{x}|\\mathbf{z})$', '#fadbd8'),
    (6.5, 1, '$\\hat{\\mathbf{x}}$', '#abebc6'),
]
for bx, by, label, color in boxes:
    ax.add_patch(plt.Rectangle((bx - 0.35, by - 0.3), 0.7, 0.6,
                                facecolor=color, edgecolor='k', linewidth=1.2,
                                zorder=2, clip_on=False))
    ax.text(bx, by, label, ha='center', va='center', fontsize=10,
            fontweight='bold', zorder=3)

# Arrows
arrow_kw = dict(arrowstyle='->', color='#555', lw=1.8, mutation_scale=15)
for (x0, y0), (x1, y1) in [
    ((0.35, 1), (1.15, 1)),
    ((1.85, 1.2), (2.65, 1.65)),
    ((1.85, 0.8), (2.65, 0.35)),
    ((3.35, 1.55), (3.85, 1.15)),
    ((3.35, 0.45), (3.85, 0.85)),
    ((4.55, 1), (5.15, 1)),
    ((5.85, 1), (6.15, 1)),
]:
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), **arrow_kw, zorder=1))

# Noise injection
ax.text(4.2, -0.1, '$\\boldsymbol{\\epsilon} \\sim \\mathcal{N}(0,I)$',
        ha='center', fontsize=10, color='crimson', style='italic')
ax.annotate('', xy=(4.2, 0.7), xytext=(4.2, 0.1),
            arrowprops=dict(arrowstyle='->', color='crimson', lw=1.5))
ax.text(4.2, 0.45, 'reparam.', ha='center', fontsize=8, color='crimson')

# KL label
ax.annotate('$D_{\\mathrm{KL}}$', xy=(3, 2.2), fontsize=11, color='#8e44ad',
            fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#f4ecf7'))
ax.annotate('', xy=(3, 2.0), xytext=(3, 1.35),
            arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=1.5, linestyle='--'))

plt.tight_layout()
plt.show()

## 3.4 Interactive VAE on 2-D data

We train a VAE on the same noisy circle data where the plain autoencoder struggled (see [Architectures Overview, Section 6](dnn_architectures_overview.ipynb)). The VAE's latent space will be **smooth and sampleable**, unlike the plain autoencoder.

In [ ]:
# @title VAE — Hyperparameters
latent_dim_vae = 2    # @param {type: "integer"} -- try 1 and 2
hidden_vae = 64       # @param {type: "integer"}
kl_weight = 1.0       # @param {type: "number"} -- try 0.1, 1.0, 5.0
n_epochs_vae = 200    # @param {type: "integer"}

# --- Encoder ---
enc_input = keras.Input(shape=(2,))
h = layers.Dense(hidden_vae, activation='relu')(enc_input)
h = layers.Dense(hidden_vae, activation='relu')(h)
z_mean = layers.Dense(latent_dim_vae, name='z_mean')(h)
z_log_var = layers.Dense(latent_dim_vae, name='z_log_var')(h)

# Reparameterization trick
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = keras.random.normal(shape=ops.shape(z_mean))
        return z_mean + ops.exp(0.5 * z_log_var) * eps

z = Sampling()([z_mean, z_log_var])
encoder_vae = keras.Model(enc_input, [z_mean, z_log_var, z], name='encoder')

# --- Decoder ---
dec_input = keras.Input(shape=(latent_dim_vae,))
h = layers.Dense(hidden_vae, activation='relu')(dec_input)
h = layers.Dense(hidden_vae, activation='relu')(h)
dec_output = layers.Dense(2)(h)
decoder_vae = keras.Model(dec_input, dec_output, name='decoder')

# --- VAE (custom training step) ---
class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.kl_weight = kl_weight
        self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with jax.default_device(jax.devices()[0]):
            pass
        x = data
        # Forward
        z_mean, z_log_var, z = self.encoder(x)
        x_recon = self.decoder(z)
        # Losses
        recon_loss = ops.mean(ops.sum(ops.square(x - x_recon), axis=1))
        kl_loss = -0.5 * ops.mean(ops.sum(
            1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var), axis=1))
        total_loss = recon_loss + self.kl_weight * kl_loss
        return {'total_loss': total_loss, 'recon_loss': recon_loss, 'kl_loss': kl_loss}

    def call(self, x):
        z_mean, z_log_var, z = self.encoder(x)
        return self.decoder(z)

vae = VAE(encoder_vae, decoder_vae, kl_weight=kl_weight)
vae.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss=None, jit_compile=False)
history_vae = vae.fit(X_circle, epochs=n_epochs_vae, batch_size=64, verbose=0)

print(f"Final total loss: {history_vae.history['total_loss'][-1]:.4f}")
print(f"  Recon loss: {history_vae.history['recon_loss'][-1]:.4f}")
print(f"  KL loss:    {history_vae.history['kl_loss'][-1]:.4f}")

In [ ]:
# Visualize: real data, VAE reconstructions, and samples from the prior
z_mean_vals, z_log_var_vals, z_vals = encoder_vae.predict(X_circle, verbose=0)
z_mean_vals = np.array(z_mean_vals)
z_vals = np.array(z_vals)
X_recon_vae = np.array(decoder_vae.predict(z_vals, verbose=0))

# Sample from the prior p(z) = N(0, I)
z_prior = np.random.randn(500, latent_dim_vae).astype(np.float32)
X_gen_vae = np.array(decoder_vae.predict(z_prior, verbose=0))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].scatter(X_circle[:, 0], X_circle[:, 1], c=theta, cmap='hsv',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[0].set_title('Real Data', fontweight='bold')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_recon_vae[:, 0], X_recon_vae[:, 1], c=theta, cmap='hsv',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[1].set_title('VAE Reconstructions', fontweight='bold')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

axes[2].scatter(X_gen_vae[:, 0], X_gen_vae[:, 1], c='#3498db',
                s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[2].set_title('Generated Samples (from prior)', fontweight='bold')
axes[2].set_aspect('equal'); axes[2].grid(True, alpha=0.3)

if latent_dim_vae == 2:
    axes[3].scatter(z_mean_vals[:, 0], z_mean_vals[:, 1], c=theta, cmap='hsv',
                    s=12, edgecolors='k', linewidths=0.3, alpha=0.7)
    # Draw unit circle (prior support)
    t_circ = np.linspace(0, 2*np.pi, 100)
    for r in [1, 2]:
        axes[3].plot(r*np.cos(t_circ), r*np.sin(t_circ), 'k--', alpha=0.3, linewidth=0.8)
    axes[3].set_title('Latent Space $\\mathbf{z}$', fontweight='bold')
    axes[3].set_xlabel('$z_1$'); axes[3].set_ylabel('$z_2$')
else:
    axes[3].scatter(z_mean_vals[:, 0], np.zeros_like(z_mean_vals[:, 0]),
                    c=theta, cmap='hsv', s=12, edgecolors='k', linewidths=0.3)
    axes[3].set_title('Latent Space (1-D)', fontweight='bold')
    axes[3].set_xlabel('$z$')
axes[3].set_aspect('equal'); axes[3].grid(True, alpha=0.3)

plt.suptitle('VAE: Training on Noisy Circle Data', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Key observation:** Unlike the plain autoencoder, the VAE can **generate** new data by sampling $\mathbf{z} \sim \mathcal{N}(0, I)$ and decoding. The KL regularizer ensures the latent space is smooth — nearby points in $\mathbf{z}$-space decode to nearby points in $\mathbf{x}$-space.

**Experiment:**
- Set `kl_weight = 0.01` (weak regularization). What happens to the generated samples? The latent space may develop holes.
- Set `kl_weight = 10.0` (strong regularization). The reconstructions become blurry — the model is forced to "forget" details to match the prior. This is the **reconstruction vs. regularization tradeoff**.
- Try `latent_dim_vae = 1`. Can the VAE represent the circle in a 1-D latent space?

In [ ]:
# Training curves: reconstruction vs. KL loss
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_vae.history['recon_loss'], color='#3498db', linewidth=2, label='Reconstruction')
axes[0].plot(history_vae.history['kl_loss'], color='#e74c3c', linewidth=2, label='KL divergence')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('VAE Training: Two Competing Objectives', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_vae.history['total_loss'], color='#2c3e50', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ELBO (negated)')
axes[1].set_title('Total Loss (Recon + KL)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# 4. Autoregressive Models and Large Language Models

## 4.1 The chain rule of probability

There is a completely different strategy for defining $p_\theta(\mathbf{x})$ that avoids latent variables altogether. For any sequence $\mathbf{x} = (x_1, x_2, \dots, x_T)$, the **chain rule of probability** gives an exact factorization:

$$\boxed{p(\mathbf{x}) = \prod_{t=1}^{T} p(x_t \mid x_1, \dots, x_{t-1}).}$$

This is not an approximation — it is an identity. The modeling choice is in how we parameterize each conditional $p_\theta(x_t \mid x_{<t})$.

**This single equation explains the entire LLM paradigm:**
- **Training:** given a text corpus, maximize $\sum_t \log p_\theta(x_t \mid x_{<t})$ — this is the **next-token prediction** objective.
- **Generation:** sample $x_1 \sim p_\theta(x_1)$, then $x_2 \sim p_\theta(x_2 \mid x_1)$, then $x_3 \sim p_\theta(x_3 \mid x_1, x_2)$, and so on.

## 4.2 Temperature sampling

When sampling from $p_\theta(x_t \mid x_{<t})$, a **temperature** parameter $\tau > 0$ controls the sharpness of the distribution:

$$p_\tau(x_t = v \mid x_{<t}) = \frac{\exp(\text{logit}_v / \tau)}{\sum_{v'} \exp(\text{logit}_{v'} / \tau)}.$$

- $\tau \to 0$: **greedy** (always picks the most likely token) — deterministic but repetitive.
- $\tau = 1$: **standard** sampling from the learned distribution.
- $\tau > 1$: **creative** / high-entropy — more diverse but less coherent.

In [ ]:
# Visualize temperature sampling on a toy vocabulary
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'by']
logits = np.array([2.5, 1.8, 0.3, 0.1, -0.2, 1.2, -0.5, -1.0])

def softmax_temp(logits, tau):
    e = np.exp(logits / tau)
    return e / e.sum()

temperatures = [0.3, 0.7, 1.0, 2.0]
colors_temp = ['#2c3e50', '#e74c3c', '#3498db', '#f39c12']

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for i, tau in enumerate(temperatures):
    probs = softmax_temp(logits, tau)
    bars = axes[i].bar(range(len(vocab)), probs, color=colors_temp[i],
                       edgecolor='k', linewidth=0.5)
    axes[i].set_xticks(range(len(vocab)))
    axes[i].set_xticklabels(vocab, rotation=45, fontsize=9)
    axes[i].set_title(f'$\\tau = {tau}$', fontweight='bold', fontsize=13)
    axes[i].set_ylim(0, 0.75)
    axes[i].grid(True, alpha=0.3, axis='y')
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    axes[i].text(0.95, 0.92, f'H = {entropy:.2f}', transform=axes[i].transAxes,
                 ha='right', fontsize=10, style='italic')

axes[0].set_ylabel('Probability')
plt.suptitle('Temperature Sampling: Controlling Diversity in Autoregressive Generation',
             fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 4.3 Toy autoregressive model: learning a sequence distribution

To see the chain-rule factorization in action, we train a small autoregressive model to learn the distribution of **binary sequences** with a hidden pattern: sequences where the bits tend to alternate (e.g., `0 1 0 1 0 1`). The model must learn the conditional probabilities $p(x_t \mid x_{<t})$ that produce this pattern.

In [ ]:
# @title Autoregressive Model — Hyperparameters
ar_seq_len = 12       # @param {type: "integer"}
ar_hidden = 32        # @param {type: "integer"}
n_epochs_ar = 80      # @param {type: "integer"}
ar_temperature = 1.0  # @param {type: "number"} -- try 0.3, 1.0, 2.0

# Generate training data: sequences that tend to alternate
def make_alternating_data(n, T, flip_prob=0.15):
    # Start with perfect alternation, then flip each bit with probability flip_prob
    X = np.zeros((n, T), dtype=np.float32)
    for i in range(n):
        X[i, 0] = np.random.randint(0, 2)
        for t in range(1, T):
            if np.random.rand() < flip_prob:
                X[i, t] = X[i, t-1]  # same as previous (break alternation)
            else:
                X[i, t] = 1 - X[i, t-1]  # alternate
    return X

X_ar = make_alternating_data(5000, ar_seq_len)

# Inputs: x_{<t} (all but last); Targets: x_t (all but first)
X_input = X_ar[:, :-1].reshape(-1, ar_seq_len - 1, 1)
y_target = X_ar[:, 1:]

# Causal model: use a 1-D causal convolution so each output depends only on past inputs
model_ar = keras.Sequential([
    layers.Input(shape=(ar_seq_len - 1, 1)),
    layers.Conv1D(ar_hidden, 3, padding='causal', activation='relu'),
    layers.Conv1D(ar_hidden, 3, padding='causal', activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model_ar.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_ar = model_ar.fit(X_input, y_target, epochs=n_epochs_ar,
                          batch_size=128, verbose=0)

print(f"Final accuracy: {history_ar.history['accuracy'][-1]:.3f}")
print(f"Parameters: {model_ar.count_params()}")

In [ ]:
# Generate sequences autoregressively with temperature
def generate_ar(model, n_seqs, seq_len, temperature=1.0):
    seqs = np.zeros((n_seqs, seq_len), dtype=np.float32)
    seqs[:, 0] = np.random.randint(0, 2, n_seqs).astype(np.float32)
    for t in range(1, seq_len):
        inp = seqs[:, :t].reshape(n_seqs, t, 1)
        # Pad to match training input length
        if t < seq_len - 1:
            pad_len = (seq_len - 1) - t
            inp = np.pad(inp, ((0, 0), (pad_len, 0), (0, 0)))
        probs = np.array(model.predict(inp, verbose=0))[:, -1, 0]
        # Temperature scaling (in logit space)
        logits = np.log(probs / (1 - probs + 1e-8) + 1e-8)
        probs_t = 1 / (1 + np.exp(-logits / temperature))
        seqs[:, t] = (np.random.rand(n_seqs) < probs_t).astype(np.float32)
    return seqs

generated = generate_ar(model_ar, 200, ar_seq_len, temperature=ar_temperature)

# Compare statistics: fraction of alternating pairs
def alt_fraction(seqs):
    return np.mean(seqs[:, 1:] != seqs[:, :-1])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Show sample sequences
for idx, (data, title) in enumerate([
    (X_ar[:10], 'Training Data'), (generated[:10], f'Generated ($\\tau$={ar_temperature})')
]):
    axes[idx].imshow(data, aspect='auto', cmap='Blues', interpolation='nearest')
    axes[idx].set_xlabel('Position $t$'); axes[idx].set_ylabel('Sequence')
    axes[idx].set_title(title, fontweight='bold')

# Alternation rate comparison
temps = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
alt_rates = [alt_fraction(generate_ar(model_ar, 200, ar_seq_len, t)) for t in temps]
axes[2].bar(range(len(temps)), alt_rates, color='steelblue', edgecolor='k')
axes[2].axhline(alt_fraction(X_ar), color='#e74c3c', linestyle='--', linewidth=2,
                label=f'Training data: {alt_fraction(X_ar):.2f}')
axes[2].set_xticks(range(len(temps)))
axes[2].set_xticklabels([f'$\\tau$={t}' for t in temps])
axes[2].set_ylabel('Alternation Rate')
axes[2].set_title('Effect of Temperature', fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print(f'Training data alternation rate: {alt_fraction(X_ar):.3f}')
print(f'Generated (tau={ar_temperature}) alternation rate: {alt_fraction(generated):.3f}')

**Experiment:**
- Change `ar_temperature`: low values ($\tau = 0.3$) produce nearly perfect alternation; high values ($\tau = 2.0$) produce near-random sequences.
- Change `flip_prob` in the data generator to make the pattern weaker or stronger. Can the model still learn it?
- This is *exactly* how LLMs work, just with a larger vocabulary (50,000+ tokens instead of 2) and a Transformer instead of causal convolutions.

## 4.4 From toy models to LLMs

A **Large Language Model** (GPT, LLaMA, etc.) is simply a scaled-up autoregressive model where:

| Component | Our toy model | LLM |
|---|---|---|
| Vocabulary size | 2 (binary) | 50,000–100,000 tokens |
| Sequence length | 12 | 2,000–128,000+ |
| Architecture | Causal Conv1D | Transformer (causal self-attention) |
| Parameters | ~3,000 | 1B–400B+ |
| Training data | 5,000 synthetic sequences | Trillions of tokens from the internet |

The mathematical structure is **identical**: the model learns $p_\theta(x_t \mid x_{<t})$ for every position $t$, and generates text by sampling one token at a time.

The **Transformer's self-attention** (see [Architectures Overview, Section 5](dnn_architectures_overview.ipynb)) is the key architectural choice. Its all-to-all attention allows each predicted token to depend on the *entire* preceding context, not just a fixed-width window. This is what makes LLMs capable of long-range coherence — and is also why their memory mechanism (the **KV cache**) is so important, a topic we will explore in the next lecture.

---
# 5. Comparative Summary

| | Probabilistic PCA | VAE | Autoregressive (LLM) |
|---|---|---|---|
| **Latent variables?** | Yes ($\mathbf{z}$) | Yes ($\mathbf{z}$) | No |
| **Decoder** | Linear ($W\mathbf{z} + \mu$) | Nonlinear neural net | $p_\theta(x_t \mid x_{<t})$ per step |
| **Training objective** | Max likelihood (closed form) | ELBO (variational) | Max likelihood (teacher forcing) |
| **Exact likelihood?** | Yes | Lower bound only | Yes (but expensive: $O(T)$ forward passes) |
| **Generation** | $\mathbf{z} \sim \mathcal{N}(0,I) \to W\mathbf{z}+\mu$ | $\mathbf{z} \sim \mathcal{N}(0,I) \to D(\mathbf{z})$ | Sequential: $x_1, x_2, \dots$ |
| **Strengths** | Simple, interpretable | Smooth latent space, fast generation | Exact factorization, scales to huge data |
| **Weaknesses** | Linear only | Blurry samples (recon. vs. KL tradeoff) | Slow generation ($O(T)$ sequential steps) |

---
# 6. Where to Go Next

This lecture surveyed the core ideas. Several important topics we did not have time to cover include:

- **Diffusion models** (DDPM, score matching): iteratively denoise Gaussian noise into data; state-of-the-art for images. The math involves stochastic differential equations and score functions $\nabla_{\mathbf{x}} \log p(\mathbf{x})$.
- **Generative Adversarial Networks (GANs)**: train a generator and discriminator in a minimax game. No explicit density, but often produces sharper images than VAEs.
- **Normalizing flows**: learn an invertible map $f: \mathbf{z} \to \mathbf{x}$ so that the change-of-variables formula gives exact likelihoods.
- **Scaling laws**: empirical power laws relating model size, dataset size, and compute to LLM performance (Kaplan et al., Hoffmann et al.).

**Next lecture:** We examine the **memory architectures** that sequence models use to store and retrieve information — from LSTM gates to the Transformer's KV cache to modern state-space models (Mamba).

---
# 7. Exercises

1. **ELBO derivation.** Starting from $\log p_\theta(\mathbf{x}) = \log \int p_\theta(\mathbf{x} | \mathbf{z})\, p(\mathbf{z})\, d\mathbf{z}$, use Jensen's inequality to derive the ELBO. Then show that the gap between $\log p_\theta(\mathbf{x})$ and the ELBO is exactly $D_{\text{KL}}(q_\phi(\mathbf{z}|\mathbf{x}) \| p_\theta(\mathbf{z}|\mathbf{x}))$.

2. **KL weight experiment.** Train the VAE with `kl_weight` set to 0.01, 0.1, 1.0, and 10.0. For each, visualize the latent space and generated samples. Explain the tradeoff you observe: why does a very small KL weight produce good reconstructions but poor samples?

3. **Temperature and entropy.** Prove that the entropy of the temperature-scaled softmax distribution $p_\tau$ is a monotonically increasing function of $\tau$. Verify empirically using the toy vocabulary example.

4. **Autoregressive generation speed.** An autoregressive model generating a sequence of length $T$ requires $T$ sequential forward passes. A VAE generates a sample in a *single* forward pass (sample $\mathbf{z}$, then decode). Discuss the tradeoffs: when is each approach preferable? What is the cost in generation quality?

5. **PCA vs. VAE on the circle.** The circle data lies on a 1-D manifold in $\mathbb{R}^2$. PCA with $k=1$ captures a line; the VAE with `latent_dim=2` captures the circle. Explain why `latent_dim=1` is insufficient even for the nonlinear VAE (hint: topology). What is the minimum latent dimension for a VAE to faithfully represent a *torus* $S^1 \times S^1$ in $\mathbb{R}^3$?